<a href="https://colab.research.google.com/github/Muffalo52/anima_lora-Colab-Trainer/blob/main/anima_lora_trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 Anima lora 고효율 학습 파이프라인 (Based on : https://github.com/sorryhyun/anima_lora)

# @markdown ### 1. 🔑 인증 설정
HF_TOKEN = "" # @param {type:"string"}
WANDB_API_KEY = "" # @param {type:"string"}

# @markdown ### 2. 📁 프로젝트 및 데이터셋 설정
# @markdown `MyDrive/lora_training/datasets/` 경로 내에 있는 폴더명 또는 zip 파일명을 입력하세요.
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "" # @param {type:"string"}

# @markdown ### 3. ⚙️ 학습 환경 설정
MIXED_PRECISION = "fp16" # @param ["bf16", "fp16"]
TARGET_COMMIT = "6398d0e4d083b478a811d6b57a5e5893582fc24e" # @param ["HEAD", "6398d0e4d083b478a811d6b57a5e5893582fc24e"]
TORCH_COMPILE = True # @param {type:"boolean"}
COMPILE_INDUCTOR_MODE = "default" # @param ["default", "reduce-overhead", "max-autotune"]
# @markdown `IS_T4_GPU`를 체크하면 코랩 무료/기본 GPU(T4) 환경에 맞춰 FlashAttention을 비활성화하고 호환 모드로 안전하게 구동합니다.
IS_T4_GPU = True # @param {type:"boolean"}
GRADIENT_CHECKPOINTING = False # @param {type:"boolean"}
# @markdown `VANILLA_LORA_MODE`를 켜면 Ortho, T-LoRA 기능을 끄고 순정 LoRA 모드로 학습합니다.
VANILLA_LORA_MODE = True # @param {type:"boolean"}
# @markdown 기존에 학습된 LoRA 파일(`.safetensors`)의 경로를 입력하면, 해당 가중치 위에서부터 새로운 학습을 시작합니다. (새로 학습할 경우 비워두세요)
NETWORK_WEIGHTS = "" # @param {type:"string"}
# @markdown 기본적으로 제외되는 레이어들을 정규식으로 지정하여 강제로 학습에 포함합니다.
# @markdown 기본적으로 학습에서 제외되는 시스템 레이어들을 어떻게 처리할지 선택합니다.
INCLUDE_MODE = "\uae30\ubcf8 (\uc81c\uc678 \uc720\uc9c0)" # @param ["기본 (제외 유지)", "Modulation 포함", "전체 포함 (모든 레이어 학습)"]
# @markdown ### 4. 🎛️ 기본 하이퍼파라미터 튜닝
TRAIN_BATCH_SIZE = 1 # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 2 # @param {type:"integer"}
# @markdown
LEARNING_RATE = "1e-4" # @param {type:"string"}
NETWORK_DIM = 32 # @param {type:"integer"}
NETWORK_ALPHA = 32 # @param {type:"integer"}
# @markdown
MAX_TRAIN_EPOCHS = 80 # @param {type:"integer"}
SAVE_EVERY_N_EPOCHS = 4 # @param {type:"integer"}
CHECKPOINTING_EPOCHS = 20 # @param {type:"integer"}
# @markdown
OPTIMIZER_TYPE = "AdamW" # @param ["AdamW", "Adafactor", "Prodigy", "prodigyplus.ProdigyPlusScheduleFree"]
LR_SCHEDULER = "cosine_with_restarts" # @param ["constant", "cosine", "cosine_with_restarts", "linear", "constant_with_warmup"]
LR_WARMUP_RATIO = 0.05 # @param {type:"slider", min:0.0, max:0.2, step:0.01}
TIMESTEP_SAMPLING = "sigmoid" #@param ["uniform", "sigmoid", "shift", "flux_shift"]
SIGMOID_SCALE = 1.2 #@param {type:"number"}
DISCRETE_FLOW_SHIFT = 1.0 # @param {type:"number"}
# @markdown
BLOCKS_TO_SWAP = 12 # @param {type:"integer"}
VAE_BATCH_SIZE = 1 # @param {type:"integer"}
VAE_CHUNK_SIZE = 64 # @param {type:"integer"}

# @markdown ### 5. 🧠 Prodigy 옵티마이저 전용 설정
prodigy_d_coef = 2.0 # @param {type:"number"}
prodigy_d0 = 1e-6 # @param {type:"number"}
prodigy_schedulefree_c = 50 # @param {type:"number"}

# @markdown ### 6. 📝 캡션 및 텍스트 증강 설정
CAPTION_DROPOUT_RATE = 0.1 # @param {type:"number"}
KEEP_TOKENS = 1 # @param {type:"integer"}

import os
import shutil
import subprocess
import re

# 경로 변수 설정
DRIVE_BASE = "/content/drive/MyDrive/lora_training"
DATASET_FOLDER = os.path.join(DRIVE_BASE, "datasets", PROJECT_NAME)
DATASET_ZIP = DATASET_FOLDER + ".zip"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output", PROJECT_NAME)
LOCAL_DATASET = "/content/anima_lora/image_dataset"

# 1. 드라이브 마운트 및 출력 폴더 생성
if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. 환경 변수 설정
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ['PYTHONWARNINGS'] = "ignore"
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

# 3 & 4. 환경 구축 및 의존성 설치
print("\n=== [1~2/5] Environment Setup and Dependencies ===")
!apt install -y pigz aria2 -qq
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv python install 3.13

REPO_URL = "https://github.com/sorryhyun/anima_lora.git"

if not os.path.exists('/content/anima_lora'):
    !git clone {REPO_URL} /content/anima_lora
os.chdir('/content/anima_lora')

if TARGET_COMMIT != "HEAD":
    print(f"체크아웃 진행 중: {TARGET_COMMIT}")
    !git checkout {TARGET_COMMIT}

if os.path.exists('.venv/bin/python'):
    print("Valid virtual environment (.venv) found. Skipping installation.")
else:
    print("Installing environment...")
    !uv venv --python 3.13 --clear
    !uv sync

!uv add wandb
!uv add prodigy-plus-schedule-free
!uv pip uninstall --python .venv -y rich

# 5. 모델 가중치 다운로드
print("\n=== [3/5] Downloading Pre-trained Models ===")
downloads = [
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors",
        "dest": "models/vae/qwen_image_vae.safetensors"
    },
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors",
        "dest": "models/text_encoders/qwen_3_06b_base.safetensors"
    },
    {
        "url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors",
        "dest": "models/diffusion_models/anima-base-v1.0.safetensors"
    }
]

for item in downloads:
    dest_path = item["dest"]
    url = item["url"]

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    if not os.path.exists(dest_path):
        print(f"⬇️ Downloading: {os.path.basename(dest_path)}")
        aria2_cmd = [
            "aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-k", "10M",
            "-d", os.path.dirname(dest_path), "-o", os.path.basename(dest_path)
        ]
        if HF_TOKEN.strip():
            aria2_cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"])
        aria2_cmd.append(url)

        try:
            subprocess.run(aria2_cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"\n💥 Error: aria2c download failed for {os.path.basename(dest_path)}. ({e})")
            raise SystemExit("다운로드에 실패하여 프로세스를 종료합니다.")
    else:
        print(f"✅ Already exists: {os.path.basename(dest_path)}")

print("✅ Model download complete.")


# 6. 데이터셋 설정
print("\n=== [4/5] Dataset Preprocessing ===")
if USE_GOOGLE_DRIVE:
    if os.path.exists(LOCAL_DATASET):
        shutil.rmtree(LOCAL_DATASET)

    if os.path.exists(DATASET_FOLDER):
        print(f"Copying folder: {DATASET_FOLDER}")
        shutil.copytree(DATASET_FOLDER, LOCAL_DATASET)
    elif os.path.exists(DATASET_ZIP):
        print(f"Extracting archive: {DATASET_ZIP}")
        shutil.unpack_archive(DATASET_ZIP, LOCAL_DATASET)
    else:
        print("Dataset not found. Skipping preprocessing.")

#warmup ratio → warmup steps
supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
images_repeats = {}
if os.path.exists(LOCAL_DATASET):
    # 최상위 폴더 확인
    top_images = [f for f in os.listdir(LOCAL_DATASET) if os.path.isfile(os.path.join(LOCAL_DATASET, f)) and f.lower().endswith(supported_types)]
    if top_images:
        images_repeats[LOCAL_DATASET] = (len(top_images), 1)

    # 하위 폴더 확인
    for item in os.listdir(LOCAL_DATASET):
        item_path = os.path.join(LOCAL_DATASET, item)
        if os.path.isdir(item_path):
            sub_images = [f for f in os.listdir(item_path) if f.lower().endswith(supported_types)]
            if sub_images:
                folder_repeats = 1
                match = re.match(r"^(\d+)_", item)
                if match:
                    folder_repeats = int(match.group(1))
                images_repeats[item_path] = (len(sub_images), folder_repeats)

pre_steps_per_epoch = sum(img * rep for (img, rep) in images_repeats.values())
steps_per_epoch = pre_steps_per_epoch / TRAIN_BATCH_SIZE if pre_steps_per_epoch > 0 else 1
actual_total_steps = int((MAX_TRAIN_EPOCHS * steps_per_epoch) / GRADIENT_ACCUMULATION_STEPS)
lr_warmup_steps = int(actual_total_steps * LR_WARMUP_RATIO)

# ------------------------------------------------------------------------------
# 훈련 파라미터 (train_args) 생성
# ------------------------------------------------------------------------------
optimizer_args_cli = ""
max_grad_norm_cli = ""

if OPTIMIZER_TYPE == "prodigyplus.ProdigyPlusScheduleFree":
    optimizer_args = ["betas=0.9,0.99", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "eps=None"]
    if prodigy_schedulefree_c > 0:
        optimizer_args.append(f"schedulefree_c={prodigy_schedulefree_c}")
    optimizer_args_cli = "--optimizer_args " + " ".join([f'"{arg}"' for arg in optimizer_args]) + " "
    max_grad_norm_cli = "--max_grad_norm 0.0 "

compile_args_cli = f"--torch_compile --compile_inductor_mode {COMPILE_INDUCTOR_MODE} " if TORCH_COMPILE else ""
t4_compat_cli = '--attn_mode "torch" ' if IS_T4_GPU else ""
grad_ckpt_cli = "--gradient_checkpointing " if GRADIENT_CHECKPOINTING else ""
unsloth_offload_checkpointing_cli = "--unsloth_offload_checkpointing " if GRADIENT_CHECKPOINTING else ""
save_precision_cli = '--save_precision "fp16" ' if IS_T4_GPU else '--save_precision "bf16" '
network_weights_cli = f'--network_weights "{NETWORK_WEIGHTS.strip()}" ' if NETWORK_WEIGHTS.strip() else ""

network_args_list = []

include_patterns_val = ""
if INCLUDE_MODE == "Modulation 포함":
    include_patterns_val = ".*_modulation.*|.*adaln_fused_down.*|.*adaln_up_.*"
elif INCLUDE_MODE == "전체 포함 (모든 레이어 학습)":
    include_patterns_val = ".*"

if include_patterns_val:
    network_args_list.append(f"include_patterns={include_patterns_val}")

network_args_cli = ""
if network_args_list:
    args_joined = " ".join([f"'{arg}'" for arg in network_args_list])
    network_args_cli = f"--network_args {args_joined} "

train_args = (
    f'--output_dir "{OUTPUT_DIR}" '
    f'--output_name "{PROJECT_NAME}" '
    f'--train_batch_size {TRAIN_BATCH_SIZE} '
    f'--network_dim {NETWORK_DIM} '
    f'--network_alpha {NETWORK_ALPHA} '
    f'--learning_rate {LEARNING_RATE} '
    f'--max_train_epochs {MAX_TRAIN_EPOCHS} '
    f'--save_every_n_epochs {SAVE_EVERY_N_EPOCHS} '
    f'--checkpointing_epochs {CHECKPOINTING_EPOCHS} '
    f'--gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} '
    f'{grad_ckpt_cli}'
    f'{unsloth_offload_checkpointing_cli}'
    f'{t4_compat_cli}'
    f'--optimizer_type "{OPTIMIZER_TYPE}" '
    f'{optimizer_args_cli}'
    f'{max_grad_norm_cli}'
    f'{network_args_cli}'
    f'{compile_args_cli}'
    f'--lr_scheduler "{LR_SCHEDULER}" '
    f'--lr_warmup_steps 0 '
    f'--lr_warmup_steps {lr_warmup_steps} '
    f'--timestep_sampling {TIMESTEP_SAMPLING} '
    f'--sigmoid_scale {SIGMOID_SCALE} '
    f'--discrete_flow_shift {DISCRETE_FLOW_SHIFT} '
    f'--blocks_to_swap {BLOCKS_TO_SWAP} '
    f'--mixed_precision "{MIXED_PRECISION}" '
    f'{save_precision_cli}'
    f'{network_weights_cli}'
    f'--caption_dropout_rate {CAPTION_DROPOUT_RATE} '
    f'--keep_tokens {KEEP_TOKENS} '
    f'--console_log_simple '
    f'--validate_every_n_epochs 9999 '
    f'--validation_split_num 0 '
    f'--sample_every_n_epochs 0 '
    f'--sample_every_n_steps 0 '
    f'--no-use_cmmd '
    f'--log_with wandb '
    f'--log_tracker_name "Anima_LoRA_Project" '
    f'--wandb_run_name "{PROJECT_NAME}_Run" '
    f'--wandb_api_key "{WANDB_API_KEY}" '
    f'--log_every_n_steps 1 '
)

# ==============================================================================
# 메인 실행 흐름
# ==============================================================================
def start_training_pipeline():
    print("\n🛠️ 내부 패치 적용 중...")
    apply_general_optimizations()
    if IS_T4_GPU:
        apply_t4_and_fp16_patches()

    apply_vanilla_lora_patch()

    print("\n🚀 Running image bucketing & VAE latent caching...")
    !uv run --no-sync make preprocess

    print("\n=== [5/5] Starting LoRA Model Training ===")
    !uv run --no-sync make lora ARGS="{train_args}"


# ==============================================================================
# 패치 코드 정의부
# ==============================================================================
def apply_vanilla_lora_patch():
    """configs/methods/lora.toml 파일을 직접 수정하여 LoRA 모드를 전환합니다."""
    lora_toml_path = "configs/methods/lora.toml"
    if os.path.exists(lora_toml_path):
        with open(lora_toml_path, "r", encoding="utf-8") as f:
            content = f.read()

        if VANILLA_LORA_MODE:
            # 순정 모드 켜기 (Ortho, T-LoRA 끄기)
            content = content.replace('use_ortho = true', 'use_ortho = false')
            content = content.replace('use_timestep_mask = true', 'use_timestep_mask = false')
            print("✅ Ortho, T-LoRA 비활성화")
        else:
            # 순정 모드 끄기 (Ortho, T-LoRA 다시 켜기)
            content = content.replace('use_ortho = false', 'use_ortho = true')
            content = content.replace('use_timestep_mask = false', 'use_timestep_mask = true')
            print("✅ Ortho, T-LoRA 활성화")

        with open(lora_toml_path, "w", encoding="utf-8") as f:
            f.write(content)

def apply_general_optimizations():
    # 1. log.py
    log_path = 'library/log.py'
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8") as f: code = f.read()
        code = code.replace('from rich.logging import RichHandler', 'raise ImportError("Force disable rich")')
        with open(log_path, "w", encoding="utf-8") as f: f.write(code)

    # 2. base.py 에포크 로그 삭제
    base_py_path = "library/datasets/base.py"
    if os.path.exists(base_py_path):
        with open(base_py_path, "r", encoding="utf-8") as f: file_content = f.read()
        target_old_code = """    def set_current_epoch(self, epoch):
        if not self.current_epoch == epoch:
            if epoch > self.current_epoch:
                num_epochs = epoch - self.current_epoch
                for _ in range(num_epochs):
                    self.current_epoch += 1
                    self.shuffle_buckets()
            else:
                self.current_epoch = epoch"""
        if target_old_code not in file_content:
            target_older_code = target_old_code.replace("num_epochs = epoch - self.current_epoch", "logger.info(\n                    \"epoch is incremented. current_epoch: {}, epoch: {}\".format(\n                        self.current_epoch, epoch\n                    )\n                )\n                num_epochs = epoch - self.current_epoch").replace("self.current_epoch = epoch", "logger.warning(\n                    \"epoch is not incremented. current_epoch: {}, epoch: {}\".format(\n                        self.current_epoch, epoch\n                    )\n                )\n                self.current_epoch = epoch")
            if target_older_code in file_content:
                file_content = file_content.replace(target_older_code, target_old_code)
                with open(base_py_path, "w", encoding="utf-8") as f: f.write(file_content)

    # 3. base.toml 강제 제어
    base_toml_path = "configs/base.toml"
    if os.path.exists(base_toml_path):
        with open(base_toml_path, "r", encoding="utf-8") as f: toml_content = f.read()
        new_toml_content = re.sub(r'validation_split_num\s*=\s*\d+', 'validation_split_num = 0', toml_content)
        new_toml_content = re.sub(r'batch_size\s*=\s*\d+', f'batch_size = {TRAIN_BATCH_SIZE}', new_toml_content)
        if not TORCH_COMPILE:
            new_toml_content = re.sub(r'torch_compile\s*=\s*true', 'torch_compile = false', new_toml_content)

        if toml_content != new_toml_content:
            with open(base_toml_path, "w", encoding="utf-8") as f: f.write(new_toml_content)

    # preprocess.toml 강제 제어
    preprocess_toml_path = "configs/preprocess.toml"
    if os.path.exists(preprocess_toml_path):
        with open(preprocess_toml_path, "r", encoding="utf-8") as f: prep_content = f.read()
        new_prep_content = re.sub(r'min_pixels\s*=\s*[\d\.]+', 'min_pixels = 0', prep_content)
        if prep_content != new_prep_content:
            with open(preprocess_toml_path, "w", encoding="utf-8") as f: f.write(new_prep_content)


    # 4. train.py WandB 로깅
    train_py_path = "train.py"
    if os.path.exists(train_py_path):
        with open(train_py_path, "r", encoding="utf-8") as f: content = f.read()
        old_single_log = """            if (
                args.optimizer_type.lower().endswith("ProdigyPlusScheduleFree".lower())
                and optimizer is not None
            ):  # tracking d*lr value of unet.
                logs["lr/d*lr"] = (
                    optimizer.param_groups[0]["d"] * optimizer.param_groups[0]["lr"]
                )"""
        new_single_log = """            if (
                args.optimizer_type.lower().endswith("prodigyplusschedulefree")
                and optimizer is not None
            ):
                logs["lr/d*lr"] = optimizer.param_groups[0]["d"] * optimizer.param_groups[0]["lr"]
                if "effective_lr" in optimizer.param_groups[0]:
                    logs["lr/effective_lr"] = optimizer.param_groups[0]["effective_lr"]
                    logs["lr/d*effective_lr"] = optimizer.param_groups[0]["d"] * optimizer.param_groups[0]["effective_lr"]"""
        old_multi_log = """                if (
                    args.optimizer_type.lower().endswith(
                        "ProdigyPlusScheduleFree".lower()
                    )
                    and optimizer is not None
                ):
                    logs[f"lr/d*lr/group{i}"] = (
                        optimizer.param_groups[i]["d"] * optimizer.param_groups[i]["lr"]
                    )"""
        new_multi_log = """                if (
                    args.optimizer_type.lower().endswith("prodigyplusschedulefree")
                    and optimizer is not None
                ):
                    logs[f"lr/d*lr/group{i}"] = optimizer.param_groups[i]["d"] * optimizer.param_groups[i]["lr"]
                    if "effective_lr" in optimizer.param_groups[i]:
                        logs[f"lr/effective_lr/group{i}"] = optimizer.param_groups[i]["effective_lr"]
                        logs[f"lr/d*effective_lr/group{i}"] = optimizer.param_groups[i]["d"] * optimizer.param_groups[i]["effective_lr"]"""
        if old_single_log in content or old_multi_log in content:
            content = content.replace(old_single_log, new_single_log).replace(old_multi_log, new_multi_log)
            with open(train_py_path, "w", encoding="utf-8") as f: f.write(content)

def apply_t4_and_fp16_patches():
    print("🚀 T4 및 FP16 최적화 패치를 시작합니다...\n")

    # 1. flash-attn 제거
    print("[1/3] flash-attn 제거 중...")
    os.system("uv remove flash-attn > /dev/null 2>&1 || true")
    os.system("uv pip uninstall --python .venv -y flash-attn > /dev/null 2>&1 || true")
    print("  ✅ 완료")

    # 4. models.py FP16 NaN 방지(Autocast) 패치
    print("\n[2/3] models.py FP16 NaN 방지 패치 중...")
    models_path = "library/anima/models.py"
    if os.path.exists(models_path):
        with open(models_path, "r", encoding="utf-8") as f: code = f.read()

        # 패치 목록: (패치 이름, 원본 코드, 바꿀 코드)
        patches = [
            (
                "RMSNorm 텐서 캐스팅",
                "    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        output = self._norm(x.float())\n        return (output * self.weight).to(x.dtype)",
                "    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != 'mps' else 'cpu'\n        with torch.autocast(device_type=device_type, dtype=torch.float32):\n            output = self._norm(x.float()).type_as(x)\n            return output * self.weight"
            ),
            (
                "FinalLayer 서명 및 FP32 적용",
                """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ):
        if self.use_adaln_lora:
            assert adaln_lora_B_T_3D is not None
            shift_B_T_D, scale_B_T_D = (
                self.adaln_modulation(emb_B_T_D)
                + adaln_lora_B_T_3D[:, :, : 2 * self.hidden_size]
            ).chunk(2, dim=-1)
        else:
            shift_B_T_D, scale_B_T_D = self.adaln_modulation(emb_B_T_D).chunk(2, dim=-1)""",
                """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ):
        device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32, enabled=use_fp32):
            if self.use_adaln_lora:
                assert adaln_lora_B_T_3D is not None
                shift_B_T_D, scale_B_T_D = (
                    self.adaln_modulation(emb_B_T_D)
                    + adaln_lora_B_T_3D[:, :, : 2 * self.hidden_size]
                ).chunk(2, dim=-1)
            else:
                shift_B_T_D, scale_B_T_D = self.adaln_modulation(emb_B_T_D).chunk(2, dim=-1)"""
            ),
            (
                "Block._forward AdaLN 청크 수정",
                """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # Compute AdaLN modulation parameters
        if self.use_adaln_lora:
            fused_down = self.adaln_fused_down(emb_B_T_D)
            down_self, down_cross, down_mlp = fused_down.chunk(3, dim=-1)
            shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                self.adaln_up_self_attn(down_self) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
            shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                self.adaln_up_cross_attn(down_cross) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
            shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                self.adaln_up_mlp(down_mlp) + adaln_lora_B_T_3D
            ).chunk(3, dim=-1)
        else:
            shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                self.adaln_modulation_self_attn(emb_B_T_D).chunk(3, dim=-1)
            )
            shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                self.adaln_modulation_cross_attn(emb_B_T_D).chunk(3, dim=-1)
            )
            shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                self.adaln_modulation_mlp(emb_B_T_D).chunk(3, dim=-1)
            )""",
                """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:
        if use_fp32:
            x_B_T_H_W_D = x_B_T_H_W_D.float()

        device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32, enabled=use_fp32):
            if self.use_adaln_lora:
                fused_down = self.adaln_fused_down(emb_B_T_D)
                down_self, down_cross, down_mlp = fused_down.chunk(3, dim=-1)
                shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                    self.adaln_up_self_attn(down_self) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
                shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                    self.adaln_up_cross_attn(down_cross) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
                shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                    self.adaln_up_mlp(down_mlp) + adaln_lora_B_T_3D
                ).chunk(3, dim=-1)
            else:
                shift_self_attn_B_T_D, scale_self_attn_B_T_D, gate_self_attn_B_T_D = (
                    self.adaln_modulation_self_attn(emb_B_T_D).chunk(3, dim=-1)
                )
                shift_cross_attn_B_T_D, scale_cross_attn_B_T_D, gate_cross_attn_B_T_D = (
                    self.adaln_modulation_cross_attn(emb_B_T_D).chunk(3, dim=-1)
                )
                shift_mlp_B_T_D, scale_mlp_B_T_D, gate_mlp_B_T_D = (
                    self.adaln_modulation_mlp(emb_B_T_D).chunk(3, dim=-1)
                )"""
            ),
            (
                "Block.forward 오프로드 체크포인트",
                """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if torch.is_grad_enabled() and self.training and self.gradient_checkpointing:
            if self.unsloth_offload_checkpointing:
                # Unsloth: async non-blocking CPU RAM offload (fastest offload method)
                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                )
            elif self.cpu_offload_checkpointing:
                # Standard cpu offload: blocking transfers
                def create_custom_forward(func):
                    def custom_forward(*inputs):
                        # Determine original device from first tensor input
                        device = next(
                            t.device for t in inputs if isinstance(t, torch.Tensor)
                        )
                        device_inputs = to_device(inputs, device)
                        outputs = func(*device_inputs)
                        return to_cpu(outputs)

                    return custom_forward

                return torch_checkpoint(
                    create_custom_forward(self._forward),
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )
            else:
                # Standard gradient checkpointing (no offload)
                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )
        else:
            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
            )""",
                """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:
        if torch.is_grad_enabled() and self.training and self.gradient_checkpointing:
            if self.unsloth_offload_checkpointing:
                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                )
            elif self.cpu_offload_checkpointing:
                def create_custom_forward(func):
                    def custom_forward(*inputs):
                        device = next(
                            t.device for t in inputs if isinstance(t, torch.Tensor)
                        )
                        device_inputs = to_device(inputs, device)
                        outputs = func(*device_inputs)
                        return to_cpu(outputs)
                    return custom_forward

                return torch_checkpoint(
                    create_custom_forward(self._forward),
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )
            else:
                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )
        else:
            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
                use_fp32,
            )"""
            ),
            (
                "Anima._run_blocks 서명 변경",
                """    def _run_blocks(
        self,
        x_padded: torch.Tensor,
        t_embedding_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params,
        capture_blocks: Optional[set] = None,
        feature_sink: Optional[dict] = None,
        stop_after_block: Optional[int] = None,
        **block_kwargs,
    ) -> torch.Tensor:""",
                """    def _run_blocks(
        self,
        x_padded: torch.Tensor,
        t_embedding_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params,
        capture_blocks: Optional[set] = None,
        feature_sink: Optional[dict] = None,
        stop_after_block: Optional[int] = None,
        use_fp32: bool = False,
        **block_kwargs,
    ) -> torch.Tensor:"""
            ),
            (
                "Anima._run_blocks 루프 내부 인자 추가",
                """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                **block_kwargs,
            )""",
                """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                use_fp32=use_fp32,
                **block_kwargs,
            )"""
            ),
            (
                "Anima.forward_mini_train_dit 메인 로직",
                """        x_B_T_H_W_D = self._run_blocks(
            x_B_T_H_W_D,
            t_embedding_B_T_D,
            crossattn_emb,
            attn_params,
            capture_blocks=return_block_features,
            feature_sink=feature_sink,
            stop_after_block=stop_after_block,
            **block_kwargs,
        )

        # Early feature-only return: skip the rest of the head entirely. The
        # captured features stay in the block-output layout (native-flatten or
        # eager grid) — consumers pool over the spatial/token axes (see the Turbo
        # PooledTokenDiscriminator), which is shape-agnostic across both.
        if return_features_early:
            return feature_sink

        # --- Native flatten: restore the original 5D shape ---
        # Delegated to a @torch.compiler.disable'd helper so the bucket-
        # dependent tuple (T_s, H_s, W_s, seq_len) never enters the compile
        # zone. See _unflatten_native_shape for rationale.
        if _native_flatten_info is not None:
            x_B_T_H_W_D = _unflatten_native_shape(x_B_T_H_W_D, _native_flatten_info)

        # Unconditional: zero buffers collapse to identity when guidance is off.
        t_emb_final = t_embedding_B_T_D + (
            self._mod_guidance_final_w * self._mod_guidance_delta
        ).unsqueeze(1)
        x_B_T_H_W_O = self.final_layer(
            x_B_T_H_W_D, t_emb_final, adaln_lora_B_T_3D=adaln_lora_B_T_3D
        )""",
                """        use_fp32 = x_B_T_H_W_D.dtype == torch.float16

        x_B_T_H_W_D = self._run_blocks(
            x_B_T_H_W_D,
            t_embedding_B_T_D,
            crossattn_emb,
            attn_params,
            capture_blocks=return_block_features,
            feature_sink=feature_sink,
            stop_after_block=stop_after_block,
            use_fp32=use_fp32,
            **block_kwargs,
        )

        # Early feature-only return: skip the rest of the head entirely. The
        # captured features stay in the block-output layout (native-flatten or
        # eager grid) — consumers pool over the spatial/token axes (see the Turbo
        # PooledTokenDiscriminator), which is shape-agnostic across both.
        if return_features_early:
            return feature_sink

        # --- Native flatten: restore the original 5D shape ---
        # Delegated to a @torch.compiler.disable'd helper so the bucket-
        # dependent tuple (T_s, H_s, W_s, seq_len) never enters the compile
        # zone. See _unflatten_native_shape for rationale.
        if _native_flatten_info is not None:
            x_B_T_H_W_D = _unflatten_native_shape(x_B_T_H_W_D, _native_flatten_info)

        # Unconditional: zero buffers collapse to identity when guidance is off.
        t_emb_final = t_embedding_B_T_D + (
            self._mod_guidance_final_w * self._mod_guidance_delta
        ).unsqueeze(1)
        x_B_T_H_W_O = self.final_layer(
            x_B_T_H_W_D, t_emb_final, adaln_lora_B_T_3D=adaln_lora_B_T_3D, use_fp32=use_fp32
        )"""
            )
        ]

        success_count = 0
        for name, old_str, new_str in patches:
            if old_str in code:
                code = code.replace(old_str, new_str)
                print(f"  ✅ [성공] {name}")
                success_count += 1
            elif new_str in code:
                print(f"  ⏭️ [스킵] {name} (이미 적용됨)")
                success_count += 1
            else:
                print(f"  ❌ [실패] {name} (원본 코드를 찾을 수 없음. 띄어쓰기나 코드가 변경되었을 수 있습니다.)")

        # 파일 저장
        with open(models_path, "w", encoding="utf-8") as f:
            f.write(code)

        if success_count == len(patches):
            print("\n🎉 models.py 모든 패치가 적용되었습니다!")
        else:
            print(f"\n⚠️ 일부 패치가 적용되지 않았습니다. ({success_count}/{len(patches)} 성공)")

    # 5. _common.py의 하드코딩된 accelerate bf16 패치
    print("\n[3/3] _common.py 정밀도 설정 패치 중...")
    common_path = "scripts/tasks/_common.py"
    if os.path.exists(common_path):
        with open(common_path, "r", encoding="utf-8") as f: code = f.read()
        MIXED_PRECISION = "fp16" # 환경에 맞게 자동 지정되도록 하려면 외부 변수를 받아오게 수정 가능
        old_common = '"--mixed_precision",\n        "bf16",'
        if old_common in code:
            code = code.replace(old_common, f'"--mixed_precision",\n        "{MIXED_PRECISION}",')
            with open(common_path, "w", encoding="utf-8") as f: f.write(code)
            print("  ✅ 적용됨: _common.py")
        elif f'"--mixed_precision",\n        "{MIXED_PRECISION}",' in code:
            print("  ⏭️ 스킵 (이미 적용됨): _common.py")
        else:
            print("  ❌ 실패: _common.py에서 대상을 찾을 수 없음")
    else:
         print(f"  ⏭️ 스킵 (파일 없음): {common_path}")

    print("\n🚀 패치 스크립트 실행 종료.")

start_training_pipeline()